# Padding & Stride — Output Size Formula

`out = floor((in + 2*pad - kernel) / stride) + 1`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Output size helper

In [ ]:
def conv_out_size(in_size, kernel, stride=1, padding=0):
    return (in_size + 2 * padding - kernel) // stride + 1

for k, s, p in [(3, 1, 0), (3, 1, 1), (3, 2, 1)]:
    print(f"in=7, k={k}, s={s}, p={p} → out={conv_out_size(7, k, s, p)}")

## 2. Visualize padding expanding input

In [ ]:
in_sz, k, pad = 5, 3, 1
padded = in_sz + 2 * pad
fig, ax = plt.subplots(figsize=(8, 2))
ax.add_patch(plt.Rectangle((0, 0.3), in_sz, 0.4, fc='steelblue', ec='black'))
ax.add_patch(plt.Rectangle((-pad, 0.2), pad, 0.6, fc='lightgray', ec='black'))
ax.add_patch(plt.Rectangle((in_sz, 0.2), pad, 0.6, fc='lightgray', ec='black'))
ax.text(in_sz/2, 0.5, f'input {in_sz}', ha='center', va='center', color='white')
ax.text(-pad/2, 0.5, 'pad', ha='center', fontsize=9)
ax.set_xlim(-1.5, in_sz+1.5); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('Padding=1 adds border pixels')
plt.tight_layout(); plt.show()

## 3. Stride effect on output length

In [ ]:
x = torch.arange(16).float().view(1, 1, 1, 16)
kernel = torch.ones(1, 1, 1, 3) / 3
outs = {}
for stride in [1, 2, 3]:
    outs[stride] = F.conv2d(x, kernel, stride=stride)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (s, o) in zip(axes, outs.items()):
    ax.plot(o.squeeze().numpy(), 'o-')
    ax.set_title(f'stride={s}, out_len={o.shape[-1]}')
plt.suptitle('Larger stride → smaller output'); plt.tight_layout(); plt.show()

## 4. Output size vs stride (chart)

In [ ]:
strides = [1, 2, 3, 4]
out_sizes = [conv_out_size(32, 3, s, 1) for s in strides]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(s) for s in strides], out_sizes, color='teal', edgecolor='black')
ax.set_xlabel('stride'); ax.set_ylabel('output spatial size')
ax.set_title('Output size for 32×32 input, k=3, pad=1')
plt.tight_layout(); plt.show()

## 5. 2D conv shape table

In [ ]:
configs = [(32, 3, 1, 1), (32, 5, 2, 2), (16, 3, 1, 0)]
labels = [f'k={k},s={s},p={p}' for _, k, s, p in configs]
sizes = [conv_out_size(inp, k, s, p) for inp, k, s, p in configs]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, sizes, color='coral', edgecolor='black')
ax.set_xlabel('output H=W'); ax.set_title('Output size for different conv configs')
plt.tight_layout(); plt.show()